# MaaS validation demo (notebook)

Short walkthrough aligned with the [validation guide](../../content/install/validation.md): configure endpoints **without** cluster admin (you provide the MaaS gateway base URL and a bearer token), create an API key, list models, send one completion, then send traffic until you get **HTTP 429**.

**Prerequisites:** Python 3.9+ (uses stdlib only — no `pip install` required).

## 1. Setup — URLs and options

Fill in the values below. **`MAAS_BASE`** is the public HTTPS origin for the MaaS gateway (same host you would use with the validation doc’s `HOST`), e.g. `https://maas.apps.example.com` — **no trailing slash**.

Paths used:
- MaaS API: `{MAAS_BASE}/maas-api/v1/...`
- Model inference uses **`MODEL_URL`** from the models list (absolute URL under the same gateway).

In [ ]:
import json
import os
import ssl
import urllib.error
import urllib.request

# --- User-provided (no cluster admin required) ---

# Public MaaS gateway base URL, e.g. https://maas.apps.cluster.example.com
MAAS_BASE = os.environ.get("MAAS_BASE", "https://maas.YOUR_DOMAIN_HERE")

# OpenShift / OAuth bearer token used only to create an API key (paste from `oc whoami -t` or console)
OPENSHIFT_TOKEN = os.environ.get("OPENSHIFT_TOKEN", "")

# MaaSSubscription name to bind the API key to (see validation.md; often e.g. simulator-subscription)
SUBSCRIPTION_NAME = os.environ.get("MAAS_SUBSCRIPTION", "simulator-subscription")

# API key metadata
API_KEY_NAME = "notebook-demo-key"
API_KEY_EXPIRES = "24h"

# Self-signed / corporate TLS: False matches curl -k; True for proper public certs
VERIFY_TLS = False

# --- Derived URLs (do not edit unless your routes differ) ---

MAAS_BASE = MAAS_BASE.rstrip("/")
API_KEYS_URL = f"{MAAS_BASE}/maas-api/v1/api-keys"
MODELS_URL = f"{MAAS_BASE}/maas-api/v1/models"

print("MAAS_BASE     :", MAAS_BASE)
print("API_KEYS_URL  :", API_KEYS_URL)
print("MODELS_URL    :", MODELS_URL)
print("SUBSCRIPTION  :", SUBSCRIPTION_NAME)
print("VERIFY_TLS    :", VERIFY_TLS)

## 2. Create an API key

Same flow as validation.md: `POST /maas-api/v1/api-keys` with your OpenShift bearer token. The **`key` is shown only once**; it is stored in **`API_KEY`** for the rest of this notebook.

If `OPENSHIFT_TOKEN` is empty, set it in the previous cell or export `OPENSHIFT_TOKEN` before launching Jupyter.

In [ ]:
from typing import Any, Dict, Optional


def http_json(
    method: str,
    url: str,
    *,
    token: Optional[str] = None,
    data: Optional[Dict[str, Any]] = None,
):
    """Minimal JSON HTTP helper (stdlib only)."""
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    body = None
    if data is not None:
        body = json.dumps(data).encode("utf-8")
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            raw = resp.read().decode("utf-8")
            return resp.status, json.loads(raw) if raw else {}
    except urllib.error.HTTPError as e:
        err_body = e.read().decode("utf-8", errors="replace")
        try:
            parsed = json.loads(err_body) if err_body else {}
        except json.JSONDecodeError:
            parsed = {"_raw": err_body}
        raise RuntimeError(f"HTTP {e.code}: {parsed}") from None


if not OPENSHIFT_TOKEN:
    raise SystemExit("Set OPENSHIFT_TOKEN in the setup cell (or environment) to create an API key.")

payload = {
    "name": API_KEY_NAME,
    "description": "MaaS notebook validation demo",
    "expiresIn": API_KEY_EXPIRES,
    "subscription": SUBSCRIPTION_NAME,
}

_, body = http_json("POST", API_KEYS_URL, token=OPENSHIFT_TOKEN, data=payload)
API_KEY = body.get("key")
if not API_KEY:
    raise RuntimeError(f"No 'key' in response: {body}")

print("API key created (prefix):", API_KEY[:20] + "…")

## 3. Model list (discovery)

`GET /maas-api/v1/models` with the API key returns models for that subscription. We capture **`MODEL_NAME`** (`id`) and **`MODEL_URL`** for the next steps.

In [ ]:
_, models_body = http_json("GET", MODELS_URL, token=API_KEY)
print(json.dumps(models_body, indent=2)[:4000])

data = models_body.get("data") or []
if not data:
    raise SystemExit("No models in response; deploy a model and check subscription binding.")

first = data[0]
MODEL_NAME = first.get("id") or first.get("name")
MODEL_URL = (first.get("url") or "").rstrip("/")
if not MODEL_NAME or not MODEL_URL:
    raise RuntimeError(f"Could not parse model id/url from: {first}")

print("\nUsing MODEL_NAME:", MODEL_NAME)
print("Using MODEL_URL :", MODEL_URL)

## 4. Single inference call

One request to **`{MODEL_URL}/v1/completions`** (same pattern as validation.md). Adjust payload if your model expects chat format.

In [ ]:
COMPLETIONS_URL = f"{MODEL_URL}/v1/completions"
inference_payload = {
    "model": MODEL_NAME,
    "prompt": "Hello from the notebook demo.",
    "max_tokens": 50,
}

status, completion = http_json("POST", COMPLETIONS_URL, token=API_KEY, data=inference_payload)
print("HTTP", status)
print(json.dumps(completion, indent=2)[:4000])

## 5. Repeat calls until HTTP 429

Sends the same style of request as validation.md’s loop, reusing **`MODEL_URL`** / **`MODEL_NAME`**. Stop on first **429** or after **`MAX_REQUESTS`** attempts.

In [ ]:
MAX_REQUESTS = 64
inference_payload = {
    "model": MODEL_NAME,
    "prompt": "Rate limit probe.",
    "max_tokens": 50,
}

ctx = ssl.create_default_context()
if not VERIFY_TLS:
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

last_code = None
for i in range(1, MAX_REQUESTS + 1):
    body = json.dumps(inference_payload).encode("utf-8")
    req = urllib.request.Request(
        COMPLETIONS_URL,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=120) as resp:
            last_code = resp.status
    except urllib.error.HTTPError as e:
        last_code = e.code
    print(f"{i:3d}  HTTP {last_code}")
    if last_code == 429:
        print("Got 429 — rate limit enforced.")
        break
else:
    print(f"No 429 after {MAX_REQUESTS} requests (quota may be high or window large).")